# 01. Search Papers — OpenAlex

OpenAlex Works API로 교육공학 주제의 영어 논문을 검색하고 `data/papers_raw.csv`로 저장합니다.


In [1]:
import os
import re
import time
import requests
import pandas as pd
from pathlib import Path
from datetime import date

DATA_DIR = Path('../data')
DATA_DIR.mkdir(parents=True, exist_ok=True)
MAILTO = os.environ.get('OPENALEX_MAILTO', 'student@example.com')
RUN_DATE = date.today().isoformat()


In [2]:
def load_topic_keywords(path: Path) -> dict:
    values = {}
    if not path.exists():
        return values
    for line in path.read_text(encoding='utf-8').splitlines():
        if '=' in line:
            k, v = line.split('=', 1)
            values[k.strip()] = v.strip()
    return values

cfg = load_topic_keywords(DATA_DIR / 'topic_keywords.txt')
TOPIC = cfg.get('TOPIC_KO', '교육공학에서 AI 기반 형성적 피드백이 자기조절학습에 미치는 영향')
RESEARCH_QUESTION = cfg.get('RESEARCH_QUESTION', 'How does AI-supported formative feedback influence self-regulated learning in educational technology environments?')
QUERY = cfg.get('QUERY', 'AI feedback self-regulated learning educational technology')
KEYWORDS = cfg.get('KEYWORDS', 'educational technology; artificial intelligence in education; AI feedback; formative feedback; self-regulated learning; learning analytics').split('; ')
FROM_YEAR = int(cfg.get('FROM_YEAR', 2020))
TO_YEAR = int(cfg.get('TO_YEAR', 2026))
SEARCH_FIELD = cfg.get('SEARCH_FIELD', 'title_and_abstract')
LANGUAGE = cfg.get('LANGUAGE', 'en')
MAX_RESULTS = 50

print('Research question:', RESEARCH_QUESTION)
print('Query:', QUERY)


Research question: How does AI-supported formative feedback influence self-regulated learning in educational technology environments?
Query: AI feedback self-regulated learning educational technology


In [3]:
def clean_text(value):
    if value is None:
        return None
    text = re.sub(r'<[^>]+>', '', str(value))
    return re.sub(r'\s+', ' ', text).strip()

def reconstruct_abstract(inv_index):
    if not inv_index:
        return None
    positions = []
    for word, idxs in inv_index.items():
        for i in idxs:
            positions.append((i, word))
    positions.sort()
    return clean_text(' '.join(w for _, w in positions))

def work_to_row(w: dict, *, query: str, run_date: str, to_year: int) -> dict:
    auths = w.get('authorships', []) or []
    head = ', '.join((a.get('author') or {}).get('display_name', '') for a in auths[:3]).strip(', ')
    authors = head + (f' et al. ({len(auths)-3} more)' if len(auths) > 3 else '')
    year = w.get('publication_year')
    cited_by = w.get('cited_by_count', 0) or 0
    age = max(to_year - (year or to_year), 1)
    primary_location = w.get('primary_location') or {}
    source = primary_location.get('source') or {}
    open_access = w.get('open_access') or {}
    return {
        'openalex_id': w.get('id'),
        'title': clean_text(w.get('title')),
        'authors': authors,
        'year': year,
        'publication_date': w.get('publication_date'),
        'venue': clean_text(source.get('display_name')),
        'type': w.get('type'),
        'cited_by_count': cited_by,
        'citations_per_year': round(cited_by / age, 2),
        'language': w.get('language'),
        'doi': w.get('doi'),
        'oa_url': open_access.get('oa_url'),
        'landing_page_url': primary_location.get('landing_page_url'),
        'is_open_access': open_access.get('is_oa'),
        'search_query': query,
        'search_run_date': run_date,
        'abstract': reconstruct_abstract(w.get('abstract_inverted_index')),
    }

def search_openalex(query, from_year, to_year, max_results=50, *, search_field='title_and_abstract', language='en'):
    base_url = 'https://api.openalex.org/works'
    params = {'mailto': MAILTO, 'sort': 'relevance_score:desc'}
    filters = [
        f'from_publication_date:{from_year}-01-01',
        f'to_publication_date:{to_year}-12-31',
    ]
    if language:
        filters.append(f'language:{language}')
    if search_field == 'title':
        filters.append(f'title.search:{query}')
    elif search_field == 'title_and_abstract':
        filters.append(f'title_and_abstract.search:{query}')
    else:
        params['search'] = query
    params['filter'] = ','.join(filters)

    rows, cursor = [], '*'
    while len(rows) < max_results:
        request_params = {**params, 'cursor': cursor, 'per-page': min(200, max_results - len(rows))}
        response = requests.get(base_url, params=request_params, timeout=30)
        response.raise_for_status()
        payload = response.json()
        for work in payload.get('results', []):
            rows.append(work_to_row(work, query=query, run_date=RUN_DATE, to_year=to_year))
            if len(rows) >= max_results:
                break
        cursor = (payload.get('meta') or {}).get('next_cursor')
        if not cursor:
            break
        time.sleep(0.1)
    return pd.DataFrame(rows)

df = search_openalex(QUERY, FROM_YEAR, TO_YEAR, MAX_RESULTS, search_field=SEARCH_FIELD, language=LANGUAGE)
print(f'{len(df)} papers fetched')
df[['title', 'year', 'cited_by_count', 'citations_per_year', 'venue']].head(10)


50 papers fetched


,title,year,cited_by_count,citations_per_year,venue
0,Beware of metacognitive laziness: Effects of g...,2024,348,174.00,British Journal of Educational Technology
1,Educational Design Principles of Using AI Chat...,2023,274,91.33,Sustainability
2,Assessment and Learning in Knowledge Spaces (A...,2021,72,14.40,Education Sciences
3,How Generative AI Influences Students’ Self-Re...,2025,71,71.00,International Journal of Engineering Pedagogy ...
4,Adapting educational practices for Generation ...,2025,39,39.00,Frontiers in Education
5,Self-Regulated Learning in the Digital Age: A ...,2025,32,32.00,The International Review of Research in Open a...
6,Enhancing legal writing skills: The impact of ...,2024,27,13.50,British Journal of Educational Technology
7,Enhancing self-regulated learning and higher-o...,2025,22,22.00,Education and Information Technologies
8,Hybrid intelligence: Human– AI coevolution and...,2025,35,35.00,British Journal of Educational Technology
9,Embedding Digital Technologies (AI and ICT) in...,2025,15,15.00,Applied Sciences


In [4]:
out = DATA_DIR / 'papers_raw.csv'
df.to_csv(out, index=False)

log = DATA_DIR / 'search_strategy.md'
log.write_text(
    f"""# Search Strategy

- run_date: {RUN_DATE}
- topic: {TOPIC}
- research_question: {RESEARCH_QUESTION}
- query: `{QUERY}`
- keywords: {', '.join(KEYWORDS)}
- source: OpenAlex Works API
- field: {SEARCH_FIELD}
- language: {LANGUAGE}
- year_range: {FROM_YEAR}-{TO_YEAR}
- max_results: {MAX_RESULTS}
- rows_saved: {len(df)}
""",
    encoding='utf-8',
)
print(f'Saved -> {out.resolve()}')
print(f'Search log -> {log.resolve()}')


Saved -> /Users/sungjae-cha/Documents/research-agent/data/papers_raw.csv
Search log -> /Users/sungjae-cha/Documents/research-agent/data/search_strategy.md
